# Accessing data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

# Holiday function

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

In [ ]:
# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

# Mean relative rank for each public holiday for every year at different time stamps
- soft coded so time stamps can be changed
- outputs a csv file
- calculates the mean, standard deviation and variance

## output csv

In [ ]:
import pandas as pd
import numpy as np
import os

def compute_yearly_relative_rank_csv(
    demand,
    holiday_lib,
    stations=["BLAKE", "PUNCH", "MEADO", "MOSMA"],
    window_days=30,
    blocks=None,
    out_csv="yearly_relative_rank_summary.csv"
):
    """
    Computes mean, standard deviation, and variance of relative rank
    for each holiday, each year, each station, across user-defined time blocks.
    Saves results to CSV.
    """

    if blocks is None:
        raise ValueError("You must supply a dictionary of time blocks.")

    demand.index = pd.to_datetime(demand.index)

    # Hourly mean demand
    hourly = demand.resample("h").mean()

    rows = []

    # Loop holidays
    for holiday_name, func in holiday_lib.items():

        # Loop years
        for year in range(2004, 2018):

            ref_date = func(year)

            # Build ±window_days window
            start = ref_date - pd.Timedelta(days=window_days)
            end   = ref_date + pd.Timedelta(days=window_days)

            window = hourly.loc[start:end].copy()
            if window.empty:
                continue

            window["date"] = window.index.date
            window["hour"] = window.index.hour

            # Compute relative rank per hour
            for station in stations:

                if station not in window.columns:
                    continue

                w = window.copy()

                w["rank"] = w.groupby("hour")[station].rank(method="average")
                n_days = w.groupby("hour")["date"].transform("nunique")
                w["relative_rank"] = w["rank"] / n_days

                # Extract holiday(Y) 24-hour profile
                expected_hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")
                holiday_day = w["relative_rank"].reindex(expected_hours)

                if holiday_day.isna().all():
                    continue

                holiday_day.index = range(24)

                # Compute block statistics
                block_stats = {}

                for block_name, hours in blocks.items():
                    values = holiday_day.loc[list(hours)]

                    block_stats[f"{block_name}_mean"] = values.mean()
                    block_stats[f"{block_name}_std"] = values.std()
                    block_stats[f"{block_name}_var"] = values.var()

                # Append row
                rows.append({
                    "holiday": holiday_name,
                    "year": year,
                    "station": station,
                    **block_stats
                })

    # Convert to DataFrame
    df = pd.DataFrame(rows)

    # Round all numeric columns to 4 decimal places
    df = df.round(4)

    # Ensure directory exists
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)

    # Save CSV
    df.to_csv(out_csv, index=False)

    return df


In [ ]:
blocks = {
    "00_04": range(0, 4),
    "04_08": range(4, 8),
    "08_15": range(8, 15),
    "15_20": range(15, 20),
    "20_24": range(20, 24),
}

df = compute_yearly_relative_rank_csv(
    demand=demand,
    holiday_lib=HOLIDAYS_VIC,
    stations=["BLAKE", "PUNCH", "MEADO", "MOSMA"],
    blocks=blocks,
    out_csv="/home/565/pv3484/aus_substation_electricity/figures/nsw_yearly_relative_rank.csv"
)


## Plotting the means

In [ ]:
# renaming x axis labels
pretty_labels = [
    "0–4",
    "4–8",
    "8–13",
    "13–20",
    "20–0"
]

### Year (x) vs Mean Relative Rank (y), one line per time block

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_year_vs_mean(df, station, holiday):
    # Filter
    sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
    sub = sub.sort_values("year")

    # Identify block mean columns
    mean_cols = [c for c in sub.columns if c.endswith("_mean")]

    # Pretty labels in the same order
    pretty_labels = ["12am–4am", "4am–8am", "8am–3pm", "3pm–8pm", "8pm–12am"]

    # Jet colormap (modern API)
    cmap = matplotlib.colormaps.get_cmap("jet").resampled(len(mean_cols))

    plt.figure(figsize=(10, 6))

    # One line per time block
    for i, col in enumerate(mean_cols):
        plt.plot(
            sub["year"],
            sub[col],
            marker="o",
            color=cmap(i),
            label=pretty_labels[i]
        )

    plt.title(f"{holiday} — {station} — Mean Relative Rank by Year")
    plt.xlabel("Year")
    plt.ylabel("Mean Relative Rank")
    plt.grid(alpha=0.3)
    plt.legend(title="Time Block", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.close()


In [ ]:
plot_year_vs_mean(df, station="BLAKE", holiday="New Year's Day")


### scatter plot of mean on x axis, time blocks on the y‑axis, one line per year

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_timeblock_means_by_year(df, station, holiday):
    sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
    sub = sub.sort_values("year")

    mean_cols = [c for c in sub.columns if c.endswith("_mean")]
    pretty_labels = ["12am–4am", "4am–8am", "8am–3pm", "3pm–8pm", "8pm–12am"]

    cmap = matplotlib.colormaps.get_cmap("jet").resampled(len(sub))

    plt.figure(figsize=(10, 6))

    for i, (_, row) in enumerate(sub.iterrows()):
        means = row[mean_cols].values

        plt.scatter(
            means,
            pretty_labels,
            color=cmap(i),
            label=row["year"]
        )

    plt.title(f"{holiday} — {station} — Mean Relative Rank by Time Block")
    plt.xlabel("Mean Relative Rank")
    plt.ylabel("Time Block")
    plt.grid(alpha=0.3)
    plt.legend(title="Year", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.close()


In [ ]:
plot_timeblock_means_by_year(df, station="BLAKE", holiday="Christmas Day")

### Time on x axis

In [ ]:
def plot_timeblock_means_flipped(df, station, holiday):
    sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
    sub = sub.sort_values("year")

    mean_cols = [c for c in sub.columns if c.endswith("_mean")]

    # Pretty labels for the x-axis
    pretty_labels = ["12am–4am", "4am–8am", "8am–3pm", "3pm–8pm", "8pm–12am"]

    plt.figure(figsize=(10, 6))

    # Use jet colormap (modern API)
    import matplotlib
    cmap = matplotlib.colormaps.get_cmap("jet").resampled(len(sub))

    for i, (_, row) in enumerate(sub.iterrows()):
        values = row[mean_cols].values
        plt.plot(pretty_labels, values, marker="o", color=cmap(i), label=row["year"])

    plt.title(f"{holiday} — {station} — Mean Relative Rank by Time Block")
    plt.xlabel("Time Block (Hour)")
    plt.ylabel("Mean Relative Rank")
    plt.legend(title="Year", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.close()


In [ ]:
plot_timeblock_means_flipped(df, station="BLAKE", holiday="New Year's Day")

### Heatmapping

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

def plot_timeblock_heatmap(df, station, holiday):
    # Filter
    sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
    sub = sub.sort_values("year")

    # Identify mean columns
    mean_cols = [c for c in sub.columns if c.endswith("_mean")]

    # Pretty labels in the same order as your blocks
    pretty_labels = ["12am–4am", "4am–8am", "8am–3pm", "3pm–8pm", "8pm–12am"]

    # Pivot: rows = years, columns = time blocks
    heat = sub[["year"] + mean_cols].set_index("year")
    heat.columns = pretty_labels
    heat = heat.sort_index(ascending=True)   # ensures 2004 → 2017

    plt.figure(figsize=(10, 6))

    # Define cmap BEFORE using it
    cmap = matplotlib.colormaps.get_cmap("viridis_r")

    sns.heatmap(
        heat,
        cmap=cmap,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        cbar_kws={"label": "Mean Relative Rank"},
        yticklabels=True
    )
    
    plt.gca().invert_yaxis()   # <-- flips the year order visually
    plt.yticks(rotation=45)

    plt.title(f"{holiday} — {station} — Mean Relative Rank Heatmap")
    plt.xlabel("Time Block")
    plt.ylabel("Year")
    plt.tight_layout()
    plt.close()


In [ ]:
plot_timeblock_heatmap(df, station="BLAKE", holiday="Australia Day")

### Mean relative ranking
- one time block per plot (year on x‑axis, mean on y‑axis)
- produce 5 plots, one for each time block

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_year_lines_per_block(df, station, holiday):
    sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
    sub = sub.sort_values("year")

    # Time block columns and pretty labels
    blocks = {
        "00_04_mean": "12am–4am",
        "04_08_mean": "4am–8am",
        "08_15_mean": "8am–3pm",
        "15_20_mean": "3pm–8pm",
        "20_24_mean": "8pm–12am"
    }

    # Jet colormap for years
    cmap = matplotlib.colormaps.get_cmap("rainbow_r").resampled(len(sub))

    # Create a row of subplots
    fig, axes = plt.subplots(
        nrows=1,
        ncols=len(blocks),
        figsize=(5 * len(blocks), 5),
        sharey=True
    )

    handles = []
    labels = []

    for ax, (col, label) in zip(axes, blocks.items()):

        # One line per year
        for j, (_, row) in enumerate(sub.iterrows()):
            h = ax.plot(
                [row["year"]],
                [row[col]],
                marker="o",
                color=cmap(j),
                label=row["year"]
            )[0]

            # Collect legend entries only once
            if row["year"] not in labels:
                handles.append(h)
                labels.append(row["year"])

        ax.set_title(label)
        ax.set_xlabel("Year")
        ax.grid(alpha=0.3)

    axes[0].set_ylabel("Mean Relative Rank")

    # Shared legend across all subplots
    fig.legend(
        handles,
        labels,
        title="Year",
        loc="upper center",
        ncol=len(labels),
        bbox_to_anchor=(0.5, 1.05)
    )

    fig.suptitle(f"{holiday} — {station}", fontsize=16, y=1.12)
    fig.tight_layout()
    plt.close()


In [ ]:
plot_year_lines_per_block(df, station="BLAKE", holiday="New Year's Day")
plot_year_lines_per_block(df, station="PUNCH", holiday="New Year's Day")
plot_year_lines_per_block(df, station="MEADO", holiday="New Year's Day")
plot_year_lines_per_block(df, station="MOSMA", holiday="New Year's Day")

### mean relative rank by year
- x axis = year
- y axis = mean relative rank

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_mean_rank_timeseries_four(df, stations, holiday):
    # Time block columns and pretty labels
    blocks = {
        "00_04_mean": "12am–4am",
        "04_08_mean": "4am–8am",
        "08_15_mean": "8am–3pm",
        "15_20_mean": "3pm–8pm",
        "20_24_mean": "8pm–12am"
    }

    # HSV colormap for the 5 time blocks
    cmap = matplotlib.colormaps.get_cmap("rainbow_r").resampled(len(blocks))

    fig, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(14, 10),
        sharey=True,
        sharex=True
    )

    axes = axes.flatten()

    handles = []
    labels = []

    for ax, station in zip(axes, stations):

        sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
        sub = sub.sort_values("year")

        # One line per time block
        for i, (col, label) in enumerate(blocks.items()):
            h = ax.plot(
                sub["year"],
                sub[col],
                marker="o",
                linewidth=2,
                color=cmap(i),
                label=label
            )[0]

            # Collect legend entries only once
            if label not in labels:
                handles.append(h)
                labels.append(label)

        ax.set_title(station)
        ax.grid(alpha=0.3)

    # Shared labels
    fig.supxlabel("Year", fontsize=12)
    fig.supylabel("Mean Relative Rank", fontsize=12)

    # Shared legend
    fig.legend(
        handles,
        labels,
        title="Time Block",
        loc="upper center",
        ncol=len(labels),
        bbox_to_anchor=(0.5, 1.02)
    )

    fig.suptitle(f"{holiday} — Mean Relative Rank by Year Across Substations", fontsize=16, y=1.08)
    fig.tight_layout()
    plt.close()


In [ ]:
stations = ["BLAKE", "PUNCH", "MEADO", "MOSMA"]

plot_mean_rank_timeseries_four(df, stations, "ANZAC Day")


### Mean relative rank
- x axis = mean relative rank
- one plot for each time step
- each line represents a year

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_rank_profile_four_stations(df, stations, holiday):
    # Time block columns and pretty labels
    blocks = [
        ("00_04_mean", "12am–4am"),
        ("04_08_mean", "4am–8am"),
        ("08_15_mean", "8am–3pm"),
        ("15_20_mean", "3pm–8pm"),
        ("20_24_mean", "8pm–12am")
    ]

    y_positions = list(range(len(blocks)))
    y_labels = [label for _, label in blocks]

    # Create figure with 4 subplots
    fig, axes = plt.subplots(
        nrows=1,
        ncols=len(stations),
        figsize=(6 * len(stations), 6),
        sharey=True
    )

    handles = []
    labels = []

    for ax, station in zip(axes, stations):

        sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
        sub = sub.sort_values("year")

        # Colour map for years
        cmap = matplotlib.colormaps.get_cmap("rainbow_r").resampled(len(sub))

        # One line per year
        for j, (_, row) in enumerate(sub.iterrows()):
            x_vals = [row[col] for col, _ in blocks]
            y_vals = y_positions

            h = ax.plot(
                x_vals,
                y_vals,
                marker="o",
                linewidth=2,
                color=cmap(j),
                label=row["year"]
            )[0]

            if row["year"] not in labels:
                handles.append(h)
                labels.append(row["year"])

        ax.set_title(station)
        ax.set_xlabel("Mean Relative Rank")
        ax.set_yticks(y_positions)
        ax.set_yticklabels(y_labels)
        ax.grid(alpha=0.3)

    axes[0].set_ylabel("Time Block")

    # Shared legend
    fig.legend(
        handles,
        labels,
        title="Year",
        loc="upper center",
        ncol=len(labels),
        bbox_to_anchor=(0.5, 1.05)
    )

    fig.suptitle(f"{holiday} — Daily Shape Profiles Across Substations", fontsize=18, y=1.12)
    fig.tight_layout()
    plt.show()


In [ ]:
stations = ["BLAKE", "PUNCH", "MEADO", "MOSMA"]

plot_rank_profile_four_stations(df, stations, "New Year's Day")


# Plotting variance 


## x axis time blocks, y axis relative rank
- one line per year
- one fig per holiday per station

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_variance_four_stations(df, stations, holiday):
    # Time block columns
    var_cols = [
        "00_04_var",
        "04_08_var",
        "08_15_var",
        "15_20_var",
        "20_24_var"
    ]

    pretty_labels = ["12am–4am", "4am–8am", "8am–3pm", "3pm–8pm", "8pm–12am"]

    # Create figure with 4 subplots (one per station)
    fig, axes = plt.subplots(
        nrows=1,
        ncols=len(stations),
        figsize=(5 * len(stations), 5),
        sharey=True
    )

    handles = []
    labels = []

    for ax, station in zip(axes, stations):

        sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
        sub = sub.sort_values("year")

        # Colour map for years
        cmap = matplotlib.colormaps.get_cmap("rainbow_r").resampled(len(sub))

        # One line per year
        for j, (_, row) in enumerate(sub.iterrows()):
            h = ax.plot(
                pretty_labels,
                row[var_cols],
                marker="o",
                color=cmap(j),
                label=row["year"]
            )[0]

            if row["year"] not in labels:
                handles.append(h)
                labels.append(row["year"])

        ax.set_title(station)
        ax.set_xlabel("Time Block")
        ax.grid(alpha=0.3)

    axes[0].set_ylabel("Variance")

    # Shared legend
    fig.legend(
        handles,
        labels,
        title="Year",
        loc="upper center",
        ncol=len(labels),
        bbox_to_anchor=(0.5, 1.05)
    )

    fig.suptitle(f"{holiday} — Variance by Time Block Across Substations", fontsize=16, y=1.12)
    fig.tight_layout()
    plt.close()


In [ ]:
stations = ["BLAKE", "PUNCH", "MEADO", "MOSMA"]

plot_variance_four_stations(df, stations, "Easter Monday")


## variance by year across substations
- x axis = year
- y axis = variance
- 4 panel plots for each substation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_variance_timeseries_four(df, stations, holiday):
    # Time block variance columns and pretty labels
    blocks = {
        "00_04_var": "12am–4am",
        "04_08_var": "4am–8am",
        "08_15_var": "8am–3pm",
        "15_20_var": "3pm–8pm",
        "20_24_var": "8pm–12am"
    }

    # HSV colormap for the 5 time blocks
    cmap = matplotlib.colormaps.get_cmap("rainbow_r").resampled(len(blocks))

    fig, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(14, 10),
        sharey=True,
        sharex=True
    )

    axes = axes.flatten()

    handles = []
    labels = []

    for ax, station in zip(axes, stations):

        sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
        sub = sub.sort_values("year")

        # One line per time block
        for i, (col, label) in enumerate(blocks.items()):
            h = ax.plot(
                sub["year"],
                sub[col],
                marker="o",
                linewidth=2,
                color=cmap(i),
                label=label
            )[0]

            if label not in labels:
                handles.append(h)
                labels.append(label)

        ax.set_title(station)
        ax.grid(alpha=0.3)

    # Shared axis labels
    fig.supxlabel("Year", fontsize=12)
    fig.supylabel("Variance", fontsize=12)

    # Shared legend
    fig.legend(
        handles,
        labels,
        title="Time Block",
        loc="upper center",
        ncol=len(labels),
        bbox_to_anchor=(0.5, 1.02)
    )

    fig.suptitle(f"{holiday} — Variance by Year Across Substations", fontsize=16, y=1.08)
    fig.tight_layout()
    plt.close()


In [ ]:
stations = ["BLAKE", "PUNCH", "MEADO", "MOSMA"]

plot_variance_timeseries_four(df, stations, "Christmas Day")

# Standard deviation

## standard deviation of the mean relative rank by year across substations
- x axis = year
- y axis = SD
- 4 panel plots for each substation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_std_timeseries_four(df, stations, holiday):
    # Time block std columns and pretty labels
    blocks = {
        "00_04_std": "12am–4am",
        "04_08_std": "4am–8am",
        "08_15_std": "8am–3pm",
        "15_20_std": "3pm–8pm",
        "20_24_std": "8pm–12am"
    }

    # HSV colormap for the 5 time blocks
    cmap = matplotlib.colormaps.get_cmap("rainbow_r").resampled(len(blocks))

    fig, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(14, 10),
        sharey=True,
        sharex=True
    )

    axes = axes.flatten()

    handles = []
    labels = []

    for ax, station in zip(axes, stations):

        sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
        sub = sub.sort_values("year")

        # One line per time block
        for i, (col, label) in enumerate(blocks.items()):
            h = ax.plot(
                sub["year"],
                sub[col],
                marker="o",
                linewidth=2,
                color=cmap(i),
                label=label
            )[0]

            if label not in labels:
                handles.append(h)
                labels.append(label)

        ax.set_title(station)
        ax.grid(alpha=0.3)

    # Shared axis labels
    fig.supxlabel("Year", fontsize=12)
    fig.supylabel("Standard Deviation", fontsize=12)

    # Shared legend
    fig.legend(
        handles,
        labels,
        title="Time Block",
        loc="upper center",
        ncol=len(labels),
        bbox_to_anchor=(0.5, 1.02)
    )

    fig.suptitle(f"{holiday} — Standard Deviation by Year Across Substations", fontsize=16, y=1.08)
    fig.tight_layout()
    plt.close()


In [ ]:
stations = ["BLAKE", "PUNCH", "MEADO", "MOSMA"]

plot_std_timeseries_four(df, stations, "New Year's Day")


# Plot mean + std and variance on 3x1 plot
- x axis = year
- one plot for each variable, all for the same substation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

def plot_mean_std_var_single(df, station, holiday):
    # Column groups
    mean_cols = {
        "00_04_mean": "12am–4am",
        "04_08_mean": "4am–8am",
        "08_15_mean": "8am–3pm",
        "15_20_mean": "3pm–8pm",
        "20_24_mean": "8pm–12am"
    }

    std_cols = {
        "00_04_std": "12am–4am",
        "04_08_std": "4am–8am",
        "08_15_std": "8am–3pm",
        "15_20_std": "3pm–8pm",
        "20_24_std": "8pm–12am"
    }

    var_cols = {
        "00_04_var": "12am–4am",
        "04_08_var": "4am–8am",
        "08_15_var": "8am–3pm",
        "15_20_var": "3pm–8pm",
        "20_24_var": "8pm–12am"
    }

    # Subset data
    sub = df[(df["station"] == station) & (df["holiday"] == holiday)]
    sub = sub.sort_values("year")

    # HSV colormap
    cmap = matplotlib.colormaps.get_cmap("rainbow_r").resampled(5)

    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=(10, 12),
        sharex=True
    )

    handles = []
    labels = []

    # Helper to plot each metric
    def plot_metric(ax, cols, title, ylabel):
        nonlocal handles, labels
        for i, (col, label) in enumerate(cols.items()):
            h = ax.plot(
                sub["year"],
                sub[col],
                marker="o",
                linewidth=2,
                color=cmap(i),
                label=label
            )[0]

            if label not in labels:
                handles.append(h)
                labels.append(label)

        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)

    # Panel 1 — Mean
    plot_metric(
        axes[0],
        mean_cols,
        f"Mean Relative Rank",
        "Mean"
    )

    # Panel 2 — Standard Deviation
    plot_metric(
        axes[1],
        std_cols,
        f"Standard Deviation",
        "Std Dev"
    )

    # Panel 3 — Variance
    plot_metric(
        axes[2],
        var_cols,
        f"Variance",
        "Variance"
    )

    # Shared x-label
    fig.supxlabel("Year", fontsize=12)

    # Shared legend
    fig.legend(
        handles,
        labels,
        title="Time Block",
        loc="upper center",
        ncol=len(labels),
        bbox_to_anchor=(0.5, 1.02)
    )

    fig.suptitle(f"{holiday} — {station} — Mean, Std, Variance", fontsize=16, y=1.05)
    fig.tight_layout()
    plt.close()


In [ ]:
for s in ["BLAKE", "PUNCH", "MEADO", "MOSMA"]:
    plot_mean_std_var_single(df, s, "Easter Sunday")